In [ ]:
# Code for BirdCLEF 2026 competition.

# A streamlined version of yesterday's code. Uses Python 3.11 so that it is compatible with TensorFlow.
# Here are the packages we will need to import.

import librosa
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# This loop counts the number of audio files in the training directory.

train_soundscapes_labels_df = pd.read_csv("train_soundscapes_labels.csv")

# We turn each 5 second increment into a separate row in the dataframe, with the corresponding labels.
# We have 1,478 files. Given that we have more than 1,000 files, we use spectograms and a CNN.
# Appends the start time to each filename.
# Because each file is 5 seconds long, we don't need to append the end time.

train_soundscapes_reduced_df = pd.DataFrame({
    "filename": train_soundscapes_labels_df.iloc[:, :2].astype(str).agg(''.join, axis=1),
    "primary_label": train_soundscapes_labels_df.iloc[:, 3]
})

print("Length of reduced dataframe:", len(train_soundscapes_reduced_df))

Length of reduced dataframe: 1478


In [18]:
print(train_soundscapes_reduced_df.head())

                                            filename  \
0  BC2026_Train_0039_S22_20211231_201500.ogg00:00:00   
1  BC2026_Train_0039_S22_20211231_201500.ogg00:00:05   
2  BC2026_Train_0039_S22_20211231_201500.ogg00:00:10   
3  BC2026_Train_0039_S22_20211231_201500.ogg00:00:15   
4  BC2026_Train_0039_S22_20211231_201500.ogg00:00:20   

                    primary_label  
0  22961;23158;24321;517063;65380  
1  22961;23158;24321;517063;65380  
2  22961;23158;24321;517063;65380  
3  22961;23158;24321;517063;65380  
4  22961;23158;24321;517063;65380  


In [4]:
# For each file, we list all animals that are present in the file
# We then classify the file as belonging to the class of each of those animals.

taxonomy_df = pd.read_csv("taxonomy.csv")
classes = ['Insecta', 'Reptilia', 'Amphibia', 'Mammalia', 'Aves']
filenames = {value: [] for value in classes}

for entry in train_soundscapes_reduced_df.itertuples():
    filename = entry.filename
    primary_label = entry.primary_label.split(';') # This records all species in a file.

    found = {value: False for value in classes}
    
    for label in primary_label:
        for value in classes:
            if taxonomy_df[taxonomy_df['primary_label'] == label]['class_name'].values[0] == value:
                found[value] = True
    
    for value in classes:
         if found[value]:
            filenames[value].append(filename)

In [5]:
# This block counts the number of files featuring each class of animal.
# Only 26 of the files are reptiles and only 346 are not amphibians.
# Because these classes are so unbalanced, we make models for them separately.

print(f"Insecta: {len(filenames['Insecta'])}")
print(f"Reptilia: {len(filenames['Reptilia'])}")
print(f"Amphibia: {len(filenames['Amphibia'])}")
print(f"Mammalia: {len(filenames['Mammalia'])}")
print(f"Aves: {len(filenames['Aves'])}")

Insecta: 336
Reptilia: 26
Amphibia: 1132
Mammalia: 82
Aves: 424


In [ ]:
NUM_CLASSES = 5
SR = 32000
N_MELS = 128
X = []
y = []

for filename in train_soundscapes_reduced_df['filename']:
    file = filename[:-8]
    start = int(filename[-2:])
    
    audio, _ = librosa.load(
        f"train_soundscapes/{file}",
        sr=SR,
        offset=start,
        duration=5
    )

    mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

    X.append(mel_db[..., np.newaxis])

    # 🔥 MULTI-LABEL TARGET
    label_vector = [0] * NUM_CLASSES
    for i, cls in enumerate(classes):
        if filename in filenames[cls]:
            label_vector[i] = 1

    y.append(label_vector)

# X is the list of spectrograms, and y is the list of multi-label vectors. We convert them to numpy arrays for use in TensorFlow.

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
def build_model(input_shape):
    model = models.Sequential()

    model.add(layers.Conv2D(16, (3,3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.3))

    # 🔥 MULTI-LABEL OUTPUT
    
    # We use sigmoid rather than softmax because we have multi-label classification.
    # Each class is independent, so we can use sigmoid to get a probability for each class.
    
    model.add(layers.Dense(NUM_CLASSES, activation='sigmoid'))

    return model

In [27]:
# Some of the classes are very unbalanced. We compute class weights to help the model learn from the underrepresented classes.

class_counts = y_train.sum(axis=0)
total = len(y_train)

class_weights = total / (NUM_CLASSES * class_counts)

print("Class weights:", class_weights)

Class weights: [ 0.8534296  11.257143    0.26092714  4.075862    0.7141994 ]


In [31]:
# bce is the loss function for multi-label classification.
# For one label, it would be Loss = -[y * log(p) + (1 - y) * log(1 - p)].
# For multiple labels, it computes the loss per class and multiplies it by the corresponding class weight.
# Also, we have binary crossentropy instead of categorical crossentropy because elements can belong to multiple class simultaneously.

import tensorflow as tf

def weighted_binary_crossentropy(weights):
    weights = tf.constant(weights, dtype=tf.float32)

    def loss(y_true, y_pred):
        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        return tf.reduce_mean(bce * weights)

    return loss

In [32]:
model = build_model(X_train.shape[1:])

model.compile(
    optimizer='adam',
    loss=weighted_binary_crossentropy(class_weights),
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop]
)

/Users/noah/tf_env/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 8s 97ms/step - accuracy: 0.5499 - loss: 0.6960 - val_accuracy: 0.5338 - val_loss: 2.5729
Epoch 2/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 7s 97ms/step - accuracy: 0.8342 - loss: 0.4725 - val_accuracy: 0.0304 - val_loss: 2.3066
Epoch 3/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 7s 92ms/step - accuracy: 0.8621 - loss: 0.4284 - val_accuracy: 0.1993 - val_loss: 1.7089
Epoch 4/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 7s 92ms/step - accuracy: 0.8697 - loss: 0.4207 - val_accuracy: 0.1993 - val_loss: 1.3708
Epoch 5/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 7s 93ms/step - accuracy: 0.8926 - loss: 0.3767 - val_accuracy: 0.1993 - val_loss: 1.2303
Epoch 6/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 7s 91ms/step - accuracy: 0.9036 - loss: 0.3772 - val_accuracy: 0.1993 - val_loss: 1.1763
Epoch 7/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 7s 91ms/step - accuracy: 0.9086 - loss: 0.3683 - val_accuracy: 0.1993 - val_loss: 1.2260
Epoch 8/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 7s 95ms/step - accuracy: 0.9086 - loss: 0.3590 - val_accuracy: 0.

In [33]:
# For each test case, we predict the probability of belonging to each class.

y_probs = model.predict(X_test, verbose=0)

In [36]:
# Each class has a different optimal threshold for converting probabilities to binary predictions.
# We find the best threshold for each class by maximizing the F1 score on the test set.
# F1 = 2 * (precision * recall) / (precision + recall)
# Precision = TP / (TP + FP)
# Recall = TP / (TP + FN)

from sklearn.metrics import f1_score
import numpy as np

best_thresholds = []

for i in range(NUM_CLASSES):
    thresholds = np.arange(0.1, 1.0, 0.05)
    best_f1 = 0
    best_t = 0.5

    for t in thresholds:
        y_pred = (y_probs[:, i] > t).astype(int)
        f1 = f1_score(y_test[:, i], y_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    best_thresholds.append(best_t)
    print(f"{classes[i]} → best threshold: {best_t:.2f}, F1: {best_f1:.3f}")

Insecta → best threshold: 0.10, F1: 1.000
Reptilia → best threshold: 0.35, F1: 1.000
Amphibia → best threshold: 0.70, F1: 0.996
Mammalia → best threshold: 0.10, F1: 0.936
Aves → best threshold: 0.30, F1: 0.930


In [37]:
# Final predictions

y_pred_final = np.zeros_like(y_probs)

for i in range(NUM_CLASSES):
    y_pred_final[:, i] = (y_probs[:, i] > best_thresholds[i]).astype(int)

In [38]:
from sklearn.metrics import classification_report

for i, cls in enumerate(classes):
    print(f"\nClass: {cls}")
    print(classification_report(y_test[:, i], y_pred_final[:, i]))


Class: Insecta
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       237
         1.0       1.00      1.00      1.00        59

    accuracy                           1.00       296
   macro avg       1.00      1.00      1.00       296
weighted avg       1.00      1.00      1.00       296


Class: Reptilia
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       291
         1.0       1.00      1.00      1.00         5

    accuracy                           1.00       296
   macro avg       1.00      1.00      1.00       296
weighted avg       1.00      1.00      1.00       296


Class: Amphibia
              precision    recall  f1-score   support

         0.0       1.00      0.97      0.99        70
         1.0       0.99      1.00      1.00       226

    accuracy                           0.99       296
   macro avg       1.00      0.99      0.99       296
weighted avg       0.99 

In [ ]:
# ChatGPT also suggested SpecAugment and Transfer Learning.
# All of the classes are strong, but Aves is the weakest.

# I am saving the model for future use.

model.save("best_multilabel_model.keras")

In [ ]:
# This block shows how to load the model for future use.

from tensorflow.keras.models import load_model

model = load_model(
    "my_model.keras",
    custom_objects={
        "loss": weighted_binary_crossentropy(class_weights)
    }
)